# Analyse des Données Météorologiques NOAA - UrbanHub (Rôle 1)

Ce notebook présente l'analyse descriptive et l'exploration des données météorologiques historiques horaires de la NOAA pour les principales villes françaises.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Chargement des données météorologiques Gold

In [2]:
gold_dir = os.path.join("data", "urbanhub-gold", "meteo")
files = glob.glob(os.path.join(gold_dir, "city=*", "daily_aggregates.parquet"))

dfs = []
for f in files:
    city = f.split("city=")[1].split(os.path.sep)[0]
    df = pd.read_parquet(f)
    df['city'] = city
    dfs.append(df)

if dfs:
    df_meteo = pd.concat(dfs, ignore_index=True)
    df_meteo['date'] = pd.to_datetime(df_meteo['date'])
    print(f"Chargement réussi : {len(df_meteo)} lignes d'agrégats météo chargées.")
    print(df_meteo.head())
else:
    print("Aucun fichier Gold météo trouvé. Exécutez le pipeline météo au préalable !")

Aucun fichier Gold météo trouvé. Exécutez le pipeline météo au préalable !


## 2. Évolution temporelle de la température

In [3]:
if dfs:
    sns.lineplot(data=df_meteo, x='date', y='avg_temperature', hue='city')
    plt.title("Évolution journalière de la température moyenne par ville")
    plt.ylabel("Température (°C)")
    plt.xlabel("Date")
    plt.tight_layout()
    plt.show()

## 3. Analyse des anomalies thermiques

In [4]:
if dfs:
    sns.boxplot(data=df_meteo, x='city', y='temp_anomaly', palette='coolwarm')
    plt.title("Distribution des anomalies de température par rapport aux normales saisonnières")
    plt.ylabel("Écart thermique (°C)")
    plt.xlabel("Ville")
    plt.tight_layout()
    plt.show()

## 4. Détection des jours extrêmes

In [5]:
if dfs:
    extremes = df_meteo.groupby('city')[['is_extreme_hot', 'is_extreme_cold', 'is_extreme_wind']].sum().reset_index()
    print("Nombre de jours extrêmes détectés par ville :")
    print(extremes)
    
    extremes.set_index('city').plot(kind='bar', stacked=True, color=['#e74c3c', '#3498db', '#f1c40f'])
    plt.title("Répartition des jours extrêmes par ville")
    plt.ylabel("Nombre de jours")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()